# Run `xme_phases`

This notebook builds HGNC-backed Phase I, Phase II, and Phase III xenobiotic metabolism gene lists from the local `xme_phases` package. It exports core and extended lists, shows summary counts, and previews key genes and references.

## 1. Locate the project and prepare output folders

Run this notebook from anywhere inside the `xme_phases` project. The cell below finds the repository root and imports the package from `src` without requiring installation.

In [1]:
from pathlib import Path
import csv
import json
import sys
from collections import Counter

try:
    from IPython.display import display
except ImportError:  # Allows this notebook's code cells to run as plain Python.
    display = print

cwd = Path.cwd().resolve()
candidates = [cwd, *cwd.parents]
PROJECT_ROOT = next((p for p in candidates if (p / "src" / "xme_phases").is_dir()), None)
if PROJECT_ROOT is None:
    raise RuntimeError("Could not find a project root containing src/xme_phases.")

src_path = str(PROJECT_ROOT / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

OUTPUT_DIR = PROJECT_ROOT / "outputs"
CACHE_DIR = PROJECT_ROOT / ".cache"
OUTPUT_DIR.mkdir(exist_ok=True)
CACHE_DIR.mkdir(exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Output folder: {OUTPUT_DIR}")
print(f"HGNC cache folder: {CACHE_DIR}")

Project root: /Users/julhashkazi/Downloads/_Enzymes/xme_phases
Output folder: /Users/julhashkazi/Downloads/_Enzymes/xme_phases/outputs
HGNC cache folder: /Users/julhashkazi/Downloads/_Enzymes/xme_phases/.cache


## 2. Import package APIs

In [2]:
from xme_phases import REFERENCES, build_xme_list, references_as_bibtex, write_table
from xme_phases.builder import summarize

print("Available reference sources:")
for ref_id, ref in REFERENCES.items():
    print(f"- {ref_id}: {ref['label']}")

Available reference sources:
- HGNC: HGNC gene symbols and gene-family membership
- XMETDB: XMetDB xenobiotic biotransformations
- BRENDA: BRENDA enzyme functional and reaction data
- CLINPGX: ClinPGx/PharmGKB pharmacogenomic annotations
- PHARMVAR: PharmVar star-allele nomenclature
- TCDB: Transporter Classification Database
- FDA_DDI: FDA drug interaction tables for CYP enzymes and transporters
- ISTRANSBASE: ISTransbase drug transporter substrates/inhibitors
- SLC_TABLES: SLC Tables solute-carrier family and member curation


## 3. Build the core and extended lists

`core` is a concise review-oriented list. `extended` adds broader CYP, ABC, and SLC/SLCO family members.

In [3]:
REFRESH_HGNC = False  # Set True to force a fresh HGNC download.

core_records = build_xme_list(tier="core", cache_dir=CACHE_DIR, refresh=REFRESH_HGNC)
extended_records = build_xme_list(tier="extended", cache_dir=CACHE_DIR, refresh=REFRESH_HGNC)

summary = {
    "core": summarize(core_records),
    "extended": summarize(extended_records),
}
print(json.dumps(summary, indent=2))

{
  "core": {
    "n_genes": 179,
    "by_phase": {
      "Phase I": 80,
      "Phase II": 61,
      "Phase III": 38
    },
    "by_family": {
      "Alcohol dehydrogenases": 8,
      "Aldehyde dehydrogenases": 19,
      "Aldo-keto reductases": 14,
      "CYP electron-transfer modifiers": 4,
      "Carboxylesterases and ester hydrolases": 6,
      "Cytochrome P450 monooxygenases, ADME/carcinogen core": 14,
      "Epoxide hydrolases": 2,
      "Flavin-containing monooxygenases": 6,
      "Inflammation-linked oxidases and peroxidases": 3,
      "Molybdo-flavoenzymes": 2,
      "Monoamine oxidases": 2,
      "Amino acid conjugation enzymes": 5,
      "Arylamine N-acetyltransferases": 2,
      "Glutathione transferases": 22,
      "Methyltransferases relevant to xenobiotic/endobiotic metabolism": 7,
      "Quinone oxidoreductases": 2,
      "Sulfotransferases": 13,
      "UDP-glucuronosyltransferases": 10,
      "ABC efflux transporters, ADME/cancer core": 11,
      "SLC/SLCO uptake and bi

## 4. Preview records

In [4]:
def rows(records):
    return [record.to_row() for record in records]

preview_columns = ["symbol", "phase", "family", "role", "reference_ids"]
core_rows = rows(core_records)

try:
    import pandas as pd
    display(pd.DataFrame(core_rows)[preview_columns].head(25))
except ImportError:
    print("\t".join(preview_columns))
    for row in core_rows[:25]:
        print("\t".join(row[col] for col in preview_columns))

,symbol,phase,family,role,reference_ids
0,ADH1A,Phase I,Alcohol dehydrogenases,alcohol oxidation; ethanol to acetaldehyde pat...,HGNC;BRENDA;CLINPGX
1,ADH1B,Phase I,Alcohol dehydrogenases,alcohol oxidation; ethanol to acetaldehyde pat...,HGNC;BRENDA;CLINPGX
2,ADH1C,Phase I,Alcohol dehydrogenases,alcohol oxidation; ethanol to acetaldehyde pat...,HGNC;BRENDA;CLINPGX
3,ADH4,Phase I,Alcohol dehydrogenases,alcohol oxidation; ethanol to acetaldehyde pat...,HGNC;BRENDA;CLINPGX
4,ADH5,Phase I,Alcohol dehydrogenases,alcohol oxidation; ethanol to acetaldehyde pat...,HGNC;BRENDA;CLINPGX
5,ADH6,Phase I,Alcohol dehydrogenases,alcohol oxidation; ethanol to acetaldehyde pat...,HGNC;BRENDA;CLINPGX
6,ADH7,Phase I,Alcohol dehydrogenases,alcohol oxidation; ethanol to acetaldehyde pat...,HGNC;BRENDA;CLINPGX
7,ADHFE1,Phase I,Alcohol dehydrogenases,alcohol oxidation; ethanol to acetaldehyde pat...,HGNC;BRENDA;CLINPGX
8,ALDH16A1,Phase I,Aldehyde dehydrogenases,aldehyde oxidation; acetaldehyde clearance and...,HGNC;BRENDA;CLINPGX
9,ALDH18A1,Phase I,Aldehyde dehydrogenases,aldehyde oxidation; acetaldehyde clearance and...,HGNC;BRENDA;CLINPGX


## 5. Check newly added coverage

This confirms that MAO genes, amino acid conjugation enzymes, BRENDA, FDA DDI, and SLC Tables are represented.

In [5]:
check_symbols = ["MAOA", "MAOB", "GLYAT", "GLYATL1", "GLYATL2", "GLYATL3", "BAAT"]
matches = [row for row in core_rows if row["symbol"] in check_symbols]

try:
    import pandas as pd
    display(pd.DataFrame(matches)[["symbol", "phase", "family", "role", "reference_ids"]])
except ImportError:
    for row in matches:
        print(f"{row['symbol']}: {row['phase']} | {row['family']} | {row['reference_ids']}")

for ref_id in ["BRENDA", "FDA_DDI", "SLC_TABLES"]:
    count = sum(ref_id in row["reference_ids"].split(";") for row in core_rows)
    print(f"{ref_id}: {count} core records")

,symbol,phase,family,role,reference_ids
0,MAOA,Phase I,Monoamine oxidases,oxidative deamination of xenobiotic and endoge...,HGNC;XMETDB;BRENDA
1,MAOB,Phase I,Monoamine oxidases,oxidative deamination of xenobiotic and endoge...,HGNC;XMETDB;BRENDA
2,BAAT,Phase II,Amino acid conjugation enzymes,"acyl-CoA amino acid N-acyl conjugation, especi...",HGNC;XMETDB;BRENDA
3,GLYAT,Phase II,Amino acid conjugation enzymes,"acyl-CoA amino acid N-acyl conjugation, especi...",HGNC;XMETDB;BRENDA
4,GLYATL1,Phase II,Amino acid conjugation enzymes,"acyl-CoA amino acid N-acyl conjugation, especi...",HGNC;XMETDB;BRENDA
5,GLYATL2,Phase II,Amino acid conjugation enzymes,"acyl-CoA amino acid N-acyl conjugation, especi...",HGNC;XMETDB;BRENDA
6,GLYATL3,Phase II,Amino acid conjugation enzymes,"acyl-CoA amino acid N-acyl conjugation, especi...",HGNC;XMETDB;BRENDA


BRENDA: 141 core records
FDA_DDI: 52 core records
SLC_TABLES: 27 core records


## 6. Summarize by phase and family

In [6]:
family_counts = Counter((record.phase, record.family) for record in core_records)
family_rows = [
    {"phase": phase, "family": family, "n_genes": n}
    for (phase, family), n in sorted(family_counts.items())
]

try:
    import pandas as pd
    display(pd.DataFrame(family_rows))
except ImportError:
    for row in family_rows:
        print(f"{row['phase']}\t{row['family']}\t{row['n_genes']}")

,phase,family,n_genes
0,Phase I,Alcohol dehydrogenases,8
1,Phase I,Aldehyde dehydrogenases,19
2,Phase I,Aldo-keto reductases,14
3,Phase I,CYP electron-transfer modifiers,4
4,Phase I,Carboxylesterases and ester hydrolases,6
5,Phase I,"Cytochrome P450 monooxygenases, ADME/carcinoge...",14
6,Phase I,Epoxide hydrolases,2
7,Phase I,Flavin-containing monooxygenases,6
8,Phase I,Inflammation-linked oxidases and peroxidases,3
9,Phase I,Molybdo-flavoenzymes,2


## 7. Export CSV, JSON, and BibTeX files

In [7]:
core_csv = write_table(core_records, OUTPUT_DIR / "xme_phase_core.csv")
core_json = write_table(core_records, OUTPUT_DIR / "xme_phase_core.json", fmt="json")
extended_csv = write_table(extended_records, OUTPUT_DIR / "xme_phase_extended.csv")
extended_json = write_table(extended_records, OUTPUT_DIR / "xme_phase_extended.json", fmt="json")

refs_bib = OUTPUT_DIR / "xme_phase_refs.bib"
refs_bib.write_text(references_as_bibtex(), encoding="utf-8")

for path in [core_csv, core_json, extended_csv, extended_json, refs_bib]:
    print(path)

/Users/julhashkazi/Downloads/_Enzymes/xme_phases/outputs/xme_phase_core.csv
/Users/julhashkazi/Downloads/_Enzymes/xme_phases/outputs/xme_phase_core.json
/Users/julhashkazi/Downloads/_Enzymes/xme_phases/outputs/xme_phase_extended.csv
/Users/julhashkazi/Downloads/_Enzymes/xme_phases/outputs/xme_phase_extended.json
/Users/julhashkazi/Downloads/_Enzymes/xme_phases/outputs/xme_phase_refs.bib


## 8. Optional: export one CSV per phase

In [8]:
for phase in ["Phase I", "Phase II", "Phase III"]:
    phase_records = [record for record in core_records if record.phase == phase]
    filename = phase.lower().replace(" ", "_") + "_core.csv"
    path = write_table(phase_records, OUTPUT_DIR / filename)
    print(f"{phase}: {len(phase_records)} genes -> {path}")

Phase I: 80 genes -> /Users/julhashkazi/Downloads/_Enzymes/xme_phases/outputs/phase_i_core.csv
Phase II: 61 genes -> /Users/julhashkazi/Downloads/_Enzymes/xme_phases/outputs/phase_ii_core.csv
Phase III: 38 genes -> /Users/julhashkazi/Downloads/_Enzymes/xme_phases/outputs/phase_iii_core.csv
